# Module 2: Simple ANN and Multi-Layer Neural Network

**Objective:** Understand how artificial neural networks work, starting from a single neuron (perceptron) and building up to multi-layer networks. We connect directly to the linear regression concepts from Module 1.

---

## Part A: The Single Neuron (Perceptron)

### 2.1 From Linear Regression to a Neuron

Recall from Module 1 that linear regression computes:

$$y = w \cdot x + b$$

A single neuron does **exactly the same thing**, but then passes the result through an **activation function**:

$$output = \sigma(w \cdot x + b)$$

Why the activation function? Without it, stacking multiple layers would still be a linear model (a composition of linear functions is linear). The activation introduces **non-linearity**, letting the network learn curves and complex boundaries.

### 2.2 Common Activation Functions

- **Sigmoid**: sigma(z) = 1 / (1 + e^(-z)) -- squashes output to (0, 1). Used for binary classification.
- **ReLU**: max(0, z) -- the most popular hidden layer activation. Simple and effective.
- **Tanh**: outputs in (-1, 1). Centered around zero, sometimes better than sigmoid.
- **Softmax**: generalizes sigmoid to multiple classes. Used in the output layer for multi-class problems.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualize activation functions
z = np.linspace(-5, 5, 200)

sigmoid = 1 / (1 + np.exp(-z))
relu = np.maximum(0, z)
tanh = np.tanh(z)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(z, sigmoid, 'b-', linewidth=2)
axes[0].set_title('Sigmoid')
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].grid(True, alpha=0.3)

axes[1].plot(z, relu, 'r-', linewidth=2)
axes[1].set_title('ReLU')
axes[1].grid(True, alpha=0.3)

axes[2].plot(z, tanh, 'g-', linewidth=2)
axes[2].set_title('Tanh')
axes[2].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[2].grid(True, alpha=0.3)

for ax in axes:
    ax.set_xlabel('z')
plt.suptitle('Activation Functions -- Adding Non-Linearity', fontsize=13)
plt.tight_layout()
plt.show()

### 2.3 A Single Neuron for Binary Classification

Task: classify points as class 0 or class 1 based on two features. A single neuron with a sigmoid activation can do this -- it's essentially **logistic regression**.

In [ ]:
# --- Single neuron from scratch ---

class SingleNeuron:
    def __init__(self, n_features, learning_rate=0.1):
        self.w = np.random.randn(n_features) * 0.01
        self.b = 0.0
        self.lr = learning_rate
        self.loss_history = []

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def forward(self, X):
        z = X @ self.w + self.b
        return self.sigmoid(z)

    def fit(self, X, y, epochs=100):
        for epoch in range(epochs):
            # Forward
            y_hat = self.forward(X)

            # Binary cross-entropy loss
            loss = -np.mean(y * np.log(y_hat + 1e-8) + (1 - y) * np.log(1 - y_hat + 1e-8))
            self.loss_history.append(loss)

            # Gradients
            error = y_hat - y
            dw = (1 / len(y)) * (X.T @ error)
            db = (1 / len(y)) * np.sum(error)

            # Update
            self.w -= self.lr * dw
            self.b -= self.lr * db

        return self

    def predict(self, X):
        return (self.forward(X) >= 0.5).astype(int)

In [ ]:
from sklearn.datasets import make_classification

# Generate linearly separable data
X_cls, y_cls = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                    n_informative=2, n_clusters_per_class=1,
                                    random_state=42)

# Train single neuron
neuron = SingleNeuron(n_features=2, learning_rate=0.1)
neuron.fit(X_cls, y_cls, epochs=200)

# Evaluate
preds = neuron.predict(X_cls)
accuracy = np.mean(preds == y_cls)
print(f'Single neuron accuracy: {accuracy:.2%}')

# Plot decision boundary
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(neuron.loss_history)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Cross-Entropy Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

# Decision boundary
xx, yy = np.meshgrid(np.linspace(X_cls[:, 0].min()-1, X_cls[:, 0].max()+1, 200),
                     np.linspace(X_cls[:, 1].min()-1, X_cls[:, 1].max()+1, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
Z = neuron.forward(grid).reshape(xx.shape)
axes[1].contourf(xx, yy, Z, levels=20, cmap='RdBu_r', alpha=0.6)
axes[1].scatter(X_cls[:, 0], X_cls[:, 1], c=y_cls, cmap='RdBu_r', edgecolors='k', linewidth=0.5)
axes[1].set_title('Decision Boundary -- Single Neuron')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part B: Multi-Layer Neural Network

### 2.4 Why Multiple Layers?

A single neuron can only draw a **straight line** as a decision boundary. Real-world data (like XOR, spirals, images) require **curved** boundaries. By stacking neurons into layers, each layer transforms the data into a new representation where the problem becomes easier.

Architecture of a 2-layer network:
- **Input Layer**: receives raw features (not learned)
- **Hidden Layer**: transforms features using weights, bias, and activation
- **Output Layer**: produces final prediction

### 2.5 Backpropagation

How do we train weights in hidden layers? We use the **chain rule of calculus** to propagate the error backward through the network. This is called **backpropagation**:

1. **Forward pass**: compute output layer by layer.
2. **Compute loss** at the output.
3. **Backward pass**: compute gradients from output to input using chain rule.
4. **Update** all weights using gradient descent.

### 2.6 Multi-Layer Network From Scratch (XOR Problem)

In [ ]:
# --- The XOR problem ---
# XOR is not linearly separable: a single neuron cannot solve it.

X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([[0], [1], [1], [0]])

print('XOR Truth Table:')
for i in range(4):
    print(f'  {X_xor[i]} -> {y_xor[i][0]}')
print()
print('A single neuron CANNOT solve this. We need a hidden layer.')

In [ ]:
class SimpleNeuralNetwork:
    """2-layer neural network: input -> hidden (with ReLU) -> output (with Sigmoid)"""

    def __init__(self, input_dim, hidden_dim, output_dim, lr=0.1):
        np.random.seed(42)
        # Xavier initialization
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, output_dim) * np.sqrt(2.0 / hidden_dim)
        self.b2 = np.zeros((1, output_dim))
        self.lr = lr
        self.loss_history = []

    def relu(self, z):
        return np.maximum(0, z)

    def relu_deriv(self, z):
        return (z > 0).astype(float)

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1        # linear transform 1
        self.a1 = self.relu(self.z1)             # activation 1
        self.z2 = self.a1 @ self.W2 + self.b2   # linear transform 2
        self.a2 = self.sigmoid(self.z2)          # output activation
        return self.a2

    def backward(self, X, y, output):
        m = X.shape[0]

        # Output layer gradient
        dz2 = output - y                                   # (m, 1)
        dW2 = (1/m) * self.a1.T @ dz2                     # (hidden, 1)
        db2 = (1/m) * np.sum(dz2, axis=0, keepdims=True)

        # Hidden layer gradient (chain rule)
        da1 = dz2 @ self.W2.T                              # (m, hidden)
        dz1 = da1 * self.relu_deriv(self.z1)               # element-wise
        dW1 = (1/m) * X.T @ dz1                            # (input, hidden)
        db1 = (1/m) * np.sum(dz1, axis=0, keepdims=True)

        # Update all weights
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def fit(self, X, y, epochs=5000):
        for epoch in range(epochs):
            output = self.forward(X)
            loss = -np.mean(y * np.log(output + 1e-8) + (1 - y) * np.log(1 - output + 1e-8))
            self.loss_history.append(loss)
            self.backward(X, y, output)

            if (epoch + 1) % 1000 == 0:
                print(f'Epoch {epoch+1:5d} | Loss: {loss:.6f}')

        return self

# Train on XOR
nn = SimpleNeuralNetwork(input_dim=2, hidden_dim=4, output_dim=1, lr=1.0)
nn.fit(X_xor, y_xor, epochs=5000)

In [ ]:
# Check predictions
predictions = nn.forward(X_xor)
print('\nXOR Predictions:')
for i in range(4):
    print(f'  Input: {X_xor[i]}  |  Predicted: {predictions[i][0]:.4f}  |  Rounded: {int(predictions[i][0] > 0.5)}  |  True: {y_xor[i][0]}')

plt.figure(figsize=(8, 5))
plt.plot(nn.loss_history)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('XOR Training -- Multi-Layer Network')
plt.grid(True, alpha=0.3)
plt.show()

### 2.7 Using Keras/TensorFlow (Production Approach)

Now let's solve a real problem with Keras. We will classify the Iris dataset (3 classes, 4 features).

In [ ]:
# Install if needed (Colab has these pre-installed)
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print(f'TensorFlow version: {tf.__version__}')

In [ ]:
# Load and prepare data
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

# Standardize features (important for neural networks)
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_iris_scaled, y_iris, test_size=0.2, random_state=42)

print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples:     {X_test.shape[0]}')
print(f'Features:         {X_train.shape[1]}')
print(f'Classes:          {len(set(y_iris))}')

In [ ]:
# Build the model
model = keras.Sequential([
    keras.layers.Dense(16, activation='relu', input_shape=(4,)),   # Hidden layer 1
    keras.layers.Dense(8, activation='relu'),                       # Hidden layer 2
    keras.layers.Dense(3, activation='softmax')                     # Output layer (3 classes)
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Train
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    verbose=0  # suppress output for clean notebook
)

# Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.2%}')

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'], label='Train')
axes[1].plot(history.history['val_accuracy'], label='Validation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.8 Key Takeaways -- Neural Networks

- A single neuron = linear regression + activation function.
- Multiple layers allow the network to learn **non-linear** decision boundaries.
- **Backpropagation** uses the chain rule to compute gradients through all layers.
- **Xavier/He initialization** prevents vanishing or exploding gradients at the start.
- **Standardizing inputs** helps the network train faster and more stably.
- Keras makes building networks simple: stack layers with Sequential, compile with an optimizer and loss, then fit.

| Concept | Module 1 (Linear Reg) | Module 2 (Neural Net) |
|---|---|---|
| Core operation | w*x + b | Same, but with activation |
| Non-linearity | None | Activation functions |
| Layers | 1 | Multiple |
| Optimization | Gradient descent | Backpropagation + GD |
| Can learn XOR? | No | Yes |